# Solutions · Chapter 03-01 · Summaries

Attempt each exercise first. E16 in particular is worth trying before reading - it has a clean answer
that ties the whole chapter together.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

late = np.array([2, 3, 3, 5, 8, 9, 12])
print("late =", late, " mean %.1f  median %.1f" % (late.mean(), np.median(late)))

## E1 · What each one solves

**The mean** solves *"find the single number whose total squared distance to the data is smallest"* -
equivalently, the balance point at which the deviations cancel.

**The median** solves *"find the single number whose total absolute distance to the data is
smallest"* - equivalently, the value with as many points above it as below.

## E2 · Squared units

Variance averages *squared* deviations, and squaring minutes gives minutes-squared. There is no
physical interpretation of a square minute, so the variance cannot be compared to the data or quoted
in a sentence. The square root undoes the squaring and returns the summary to minutes, which is why
the standard deviation is the one that appears in reports and the variance is the one that appears in
algebra.

## E3 · 12 versus 14

You used different `ddof` values - almost certainly `np.var` (`ddof=0`) against `pandas.var`
(`ddof=1`).

**Which of you should change: whoever is not treating the data as a sample.** If the seven values are
the entire population of interest, 12 is right. If they are seven observations of an ongoing process -
which is nearly always the situation - 14 is right and the person using `ddof=0` should change.

The important part of the answer is that this is a *question about what the data is*, not a
convention to be settled by preference.

## E4 · Hand calculation

In [ ]:
x = np.array([4, 4, 6, 10, 16])
dev = x - x.mean()

print("values          :", x)
print("mean            : %.1f   (sum %d / 5)" % (x.mean(), x.sum()))
print("median          : %.1f" % np.median(x))
print("deviations      :", dev)
print("squared         :", dev ** 2)
print("sum of squares  : %d" % (dev ** 2).sum())
print("variance /n     : %.2f" % ((dev ** 2).sum() / 5))
print("variance /(n-1) : %.2f" % ((dev ** 2).sum() / 4))
print("sd (ddof=1)     : %.4f" % x.std(ddof=1))

Mean 8, median 6, sum of squares 104, variance 20.80 or 26.00, standard deviation 5.0990.

Worth noticing: mean 8 against median 6, from a single large value at 16. The same shape as the
festival data in 02-05, in five numbers.

## E5 · Why deviations sum to zero

In [ ]:
print("sum of deviations from the MEAN   : %.1f" % (x - x.mean()).sum())
print("sum of deviations from the MEDIAN : %.1f" % (x - np.median(x)).sum())

**Why it holds for any dataset.** The sum of deviations is `sum(x) - n * mean`, and the mean is
*defined* as `sum(x) / n`, so the second term is `sum(x)` and the whole thing is zero. It is not a
property the data has to earn; it is the definition rearranged.

**The same does not hold for the median** - here the deviations from the median sum to **10**. The
median balances the *count* of points either side, not their total distance. That is exactly the
difference between minimising absolute error and minimising squared error, seen from the other
direction.

## E6 · Mean 12, median 7

**Shape:** right-skewed. Most transactions are small - around 7 EUR - with a tail of large ones
dragging the mean up to 12. A long right tail and a bunched left side, exactly like the shop data
in 02-05.

**Which number forecasts a day's takings from 300 transactions: the mean.** Takings are a *total*,
and total = mean x count, by definition. 300 x 12 = 3,600 EUR. Using the median gives 2,100 EUR and
would be wrong by 1,500 EUR, because the median deliberately ignores the magnitude of the large
transactions and those transactions are real money.

**The rule this illustrates:** the mean is the right summary whenever the quantity you care about is
a total. The median is the right summary when you care about a typical case. "What will we take
today" is a total; "what does a customer usually spend" is a typical case, and it is 7 EUR.

## E7 · `summary`

In [ ]:
def summary(values):
    values = np.asarray(values, dtype=float)
    mean = values.mean()
    sd = values.std(ddof=1)
    return {
        "n": len(values),
        "mean": round(float(mean), 3),
        "median": float(np.median(values)),
        "sd": round(float(sd), 3),
        "IQR": round(float(np.quantile(values, 0.75) - np.quantile(values, 0.25)), 3),
        "beyond 2 sd": int((np.abs(values - mean) > 2 * sd).sum()),
    }


corrupted = late.copy()
corrupted[-1] = 200

print("clean     :", summary(late))
print("corrupted :", summary(corrupted))

**What moved:** the mean (6.0 to 32.857) and the standard deviation (3.742 to **73.751**, a factor of
twenty). **What did not move:** the median, at 5.0, and the IQR, at 5.5 in both.

The `beyond 2 sd` count is the interesting one - it goes from 0 to 1, so on this data the rule does
flag the corrupted value. Do not take that as reassurance: 02-05's masking effect is exactly the case
where it fails, and it needs only a second bad value to appear here too. The rule caught this one
because a single outlier among seven cannot inflate the standard deviation quite enough to hide
itself.

## E8 · Does `n - 1` still work on a non-normal population?

In [ ]:
rng = np.random.default_rng(3)
scale = 2.0                      # exponential: mean = 2, variance = scale**2 = 4

rows = []
for n in [5, 10, 30]:
    samples = rng.exponential(scale, (200_000, n))
    dev = samples - samples.mean(axis=1, keepdims=True)
    total = (dev ** 2).sum(axis=1).mean()
    rows.append({"n": n, "average of /n": round(total / n, 4),
                 "average of /(n-1)": round(total / (n - 1), 4),
                 "(n-1)/n x 4": round(4 * (n - 1) / n, 4)})

print(pd.DataFrame(rows).to_string(index=False))
print("\nthe true variance of this population is exactly 4.0")

**Yes, exactly.** At n = 5 the `/n` formula averages **3.1928**, and the value it should average if
the bias factor `(n-1)/n` holds is `4 x 0.8 = 3.2000` - a match to within simulation noise. The
`n - 1` version returns 3.9910, 4.0032 and 4.0073 against a truth of 4.

**This is the point of the exercise.** The `n - 1` correction is not a normal-distribution result. It
follows from the algebra of measuring deviations from your own sample's mean, and that algebra does
not care what shape the population has. The exponential distribution is strongly skewed and it makes
no difference at all.

What *does* require normality: the two-thirds rule, the three-sigma rule, and most confidence
intervals in their textbook form. Those come in 03-02 and 03-03.

## E9 · The trimmed mean

In [ ]:
def trimmed_mean(values, fraction):
    values = np.sort(np.asarray(values, dtype=float))
    k = int(np.floor(len(values) * fraction))
    kept = values[k:len(values) - k] if len(values) - 2 * k > 0 else values
    return kept.mean(), k


rows = []
for worst in [12, 50, 1000]:
    c = late.copy()
    c[-1] = worst
    row = {"worst morning": worst, "mean": round(float(c.mean()), 2)}
    for fraction in [0.0, 0.1, 0.25]:
        value, k = trimmed_mean(c, fraction)
        row["trim %.2f (drops %d each end)" % (fraction, k)] = round(float(value), 2)
    row["median"] = float(np.median(c))
    rows.append(row)

print(pd.DataFrame(rows).to_string(index=False))

**Where it sits:** the trimmed mean is a dial between the mean (trim 0) and the median (trim towards
0.5). At 25% it drops one value from each end of seven and gives **5.60** no matter how bad the worst
morning gets - robust, and still using more of the data than the median does.

**The wrinkle worth catching:** a 10% trim on seven values drops `floor(0.7) = 0` from each end, so it
is *identical to the mean* and goes to 147.14 with everything else. The breakdown point of a trimmed
mean is set by how many points it actually removes, not by the fraction you asked for, and on small
samples those differ a lot. Asking for a 10% trim and receiving no trimming at all, silently, is the
kind of thing this course keeps finding.

## E10 · Standard deviation larger than the mean

**What it tells you about the shape:** the distribution cannot be symmetric. Response times cannot be
negative, so a mean of 250 with a spread of 400 must come from a long right tail - most requests fast,
a few extremely slow. A symmetric distribution with those numbers would put a large share of its mass
below zero, which is impossible here.

**What to ask for instead:** percentiles. The median, the 95th and the 99th - "half of requests under
120 ms, 95% under 800 ms, 99% under 3 s" is actionable and the mean-and-sd pair is not. For latency
specifically the tail *is* the product: the 99th percentile is what your unhappiest users experience,
and it is invisible in both numbers reported.

## E11 · Same mean, different spreads

**Prefer A (sd 5):** predictability. Staffing, inventory and cash flow all plan against the worst
plausible week rather than the average one, and A's worst week is much closer to its average. Low
variance is worth real money in any operation with a capacity constraint.

**Prefer B (sd 40):** upside, and information. B's spread means some periods are far above the mean -
if those are repeatable (a salesperson, a channel, a segment) then B contains a discoverable win that
A does not. It also suggests B's outcomes depend on something identifiable, which is a lead.

The general form: **low variance is preferable when you are protecting a downside, high variance when
you are shopping for an upside you can select.** Which applies depends on whether you can act on the
good cases separately.

## E12 · The two-library pipeline

In [ ]:
for n in [10, 100, 10_000]:
    ratio = np.sqrt(n / (n - 1))
    print("n = %6d   sd(ddof=1) / sd(ddof=0) = %.6f   -> %.4f%% difference"
          % (n, ratio, 100 * (ratio - 1)))

**The cause:** `np.std` defaults to `ddof=0` and `pandas.Series.std` defaults to `ddof=1`, so the
same column standardised through the two paths differs by a factor of `sqrt(n / (n - 1))`.

**The size:** **5.41%** at n = 10, and **0.005%** at n = 10,000. Small data, real problem; large data,
invisible - which is worse, because it means the bug is discovered in production on a small segment
after passing every test on the full dataset.

**The one-word fix: `ddof`.** Pass it explicitly everywhere - `np.std(x, ddof=1)` - and never rely on
either default. The general habit: when two libraries offer the same function with different
defaults, write the argument out even when it matches the default, because the next reader cannot see
which default you were relying on.

## E13 · Median or mean, in four sentences

> "I report the **median** when the number is meant to describe a typical case and the distribution
> has a tail that would drag the mean somewhere no one actually is - salaries, response times, house
> prices. I report the **mean** when the quantity I care about is a total, since total equals mean
> times count and the median has no such relationship - revenue forecasts, capacity planning, and any
> per-unit cost. The property separating them is what each minimises: the mean minimises squared
> error and so is pulled by large values, while the median minimises absolute error and is not.
> Reporting only one is misleading when they differ substantially - a mean of 12 against a median of
> 7 is itself the finding, so I give both and let the gap do the talking."

Four sentences, one concrete example per case, and the last one is what distinguishes a good answer:
it shows you know that the choice is sometimes "neither alone".

## E14 · Hospital wards

**The extra column you need: the number of patients (or discharges) per ward.**

**What goes wrong without it:** averaging the ward means unweighted answers "what is the average
ward's average stay", which weights a 6-bed specialist unit exactly as heavily as a 60-bed general
ward. If the small units have long stays - which is typical, since specialist and palliative units are
both small and slow - the hospital-wide figure comes out far too high.

**Which idea it is:** the weighted mean, and behind it 02-01's question about what one row represents.
A row is a ward, the question is about patients, and converting between the two needs the counts. It
is the same arithmetic that made the bus routes read 12.00 minutes instead of 5.70.

## E15 · For the manager

> "Pay at most companies is not spread evenly around a middle - most people are clustered fairly low
> and a handful at the top earn many times what everyone else does. When you add everyone up and
> divide by the number of staff, those few large salaries pull the answer upwards, past the point
> where most of the staff actually sit. So the 'average' ends up in a gap: too high to describe the
> many, too low to describe the few. If you want the figure that describes a normal member of staff,
> line everyone up in order of pay and take the person standing in the middle."

77 words, and it ends by defining the median without naming it - which is the version a manager can
repeat to somebody else.

## E16 · Other powers

In [ ]:
grid = np.arange(0, 14, 0.001)

rows = []
for power in [1.0, 1.5, 2.0, 4.0, 10.0, 50.0]:
    errors = np.array([(np.abs(late - g) ** power).sum() for g in grid])
    rows.append({"power": power, "minimiser": round(float(grid[errors.argmin()]), 3)})

print(pd.DataFrame(rows).to_string(index=False))
print()
print("median   %.1f" % np.median(late))
print("mean     %.1f" % late.mean())
print("midrange %.1f   (smallest + largest) / 2" % ((late.min() + late.max()) / 2))

In [ ]:
powers = np.arange(1.0, 12.01, 0.25)
minimisers = [grid[np.array([(np.abs(late - g) ** p).sum() for g in grid]).argmin()] for p in powers]

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(powers, minimisers, color="#0072B2", linewidth=2)
for value, label in [(np.median(late), "median"), (late.mean(), "mean"),
                     ((late.min() + late.max()) / 2, "midrange")]:
    ax.axhline(value, color="grey", linestyle="--", linewidth=0.9)
    ax.text(11.2, value + 0.05, label, fontsize=9, ha="right")
ax.set_xlabel("power the errors are raised to")
ax.set_ylabel("the number that minimises total error")
ax.set_title("One family, three familiar summaries")
plt.tight_layout()
plt.show()

**Power 1.5 gives 5.611 and power 4 gives 6.520** - between the median and the mean, and beyond the
mean, respectively.

The pattern is the answer: **the power controls how much attention large errors get, and therefore
how far the summary is pulled towards extreme values.**

| Power | Minimiser | Value here |
|---|---|---|
| 1 | median | 5.0 |
| 2 | mean | 6.0 |
| towards infinity | **midrange**, `(min + max) / 2` | 7.0 |

At power 50 the minimiser is already **7.000**, which is exactly `(2 + 12) / 2`. As the power grows,
the largest error dominates the sum completely, so minimising it means minimising *the worst error* -
and the number that does that is the midpoint of the range, which depends on nothing but the two most
extreme values.

That gives you the full spectrum in one line: **from the median, which ignores magnitudes entirely, to
the midrange, which ignores everything except the extremes - with the mean sitting at power 2**. The
choice of power is a choice about how much a large error should hurt, and every loss function in this
course is a point on this spectrum or a variation of one.

Two forward references: module 07 chooses between MAE (power 1) and RMSE (power 2) for exactly this
reason, and the Huber loss in module 05 is deliberately power 2 near zero and power 1 far away, to
get the differentiability of squares with the robustness of absolutes.

## Where to go next

**03-02 · Distributions and sampling.** Every number here came from seven mornings, and a different
seven would have given different numbers. The next chapter measures how much different, which turns
out to be predictable - and is what makes it possible to say whether a difference between two numbers
means anything at all.